In [13]:
import yfinance as yf

import pandas as pd
import numpy as np


import os

In [10]:
def download_sp100_data(start_date="2019-01-01", end_date="2024-01-01"):
    # Full list of S&P 100 Tickers
    tickers = [
        "AAPL", "ABBV", "ABT", "ACN", "ADBE", "AMAT", "AMD", "AMGN", "AMT", "AMZN",
        "AVGO", "AXP", "BA", "BAC", "BK", "BKNG", "BLK", "BMY", "BRK-B", "C",
        "CAT", "CL", "CMCSA", "COF", "COP", "COST", "CRM", "CSCO", "CVS", "CVX",
        "DE", "DHR", "DIS", "DUK", "EMR", "FDX", "GD", "GE", "GILD",
        "GM", "GOOG", "GOOGL", "GS", "HD", "HON", "IBM", "INTC", "INTU", "ISRG",
        "JNJ", "JPM", "KO", "LIN", "LLY", "LMT", "LOW", "LRCX", "MA", "MCD",
        "MDLZ", "MDT", "META", "MMM", "MO", "MRK", "MS", "MSFT", "MU", "NEE",
        "NFLX", "NKE", "NOW", "NVDA", "ORCL", "PEP", "PFE", "PG", "PLTR", "PM",
        "QCOM", "RTX", "SBUX", "SCHW", "SO", "SPG", "T", "TMO", "TMUS", "TSLA",
        "TXN", "UBER", "UNH", "UNP", "UPS", "USB", "V", "VZ", "WFC", "WMT", "XOM"
    ]
    
    print(f"Downloading data for {len(tickers)} tickers...")
    
    # auto_adjust=True ensures the 'Close' prices are fully adjusted for splits and dividends.
    data = yf.download(tickers, start=start_date, end=end_date, auto_adjust=True)['Close']
    
    # Save locally to avoid re-downloading
    file_path = "sp100_adj_close.csv"
    data.to_csv(file_path)
    
    print(f"Data saved to {file_path}")
    print(data.head())
    return data

def load_and_clean_sp100_data(file_path="sp100_adj_close.csv"):
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Cannot find {file_path}. Ensure it is in your project directory.")
        
    print(f"Loading local data from {file_path}...")
    
    # Load the data, setting the first column (Date) as the index
    # parse_dates=True ensures the index is treated as time-series data
    df = pd.read_csv(file_path, index_col=0, parse_dates=True)
    
    print(f"Original data shape: {df.shape}")
    
    # Drop any stocks that have missing values (NaN) 
    # This is non-negotiable for PCA; the covariance matrix must be full-rank.
    df_clean = df.dropna(axis=1)
    print(f"Shape after dropping stocks with incomplete data: {df_clean.shape}")
    
    print("Data loading and cleaning complete.")
    return df_clean

In [11]:
df = load_and_clean_sp100_data()

Loading local data from sp100_adj_close.csv...
Original data shape: (1258, 100)
Shape after dropping stocks with incomplete data: (1258, 98)
Data loading and cleaning complete.


In [15]:
# Calculate daily log returns: ln(P_t / P_{t-1})
# This transforms absolute prices into a normalized rate of change
df_log_returns = np.log(df / df.shift(1))
print(f"Log returns calculated. Shape: {df_log_returns.shape}")

# Drop the first row since the shift operation creates a row of NaNs at the start
df_log_returns = df_log_returns.dropna(axis=0)
print(f"Log returns calculated. Shape: {df_log_returns.shape}")

# Standardize the returns (Z-score)
# (Return - Mean) / Standard Deviation
df_z_log_returns = (df_log_returns - df_log_returns.mean()) / df_log_returns.std()

Log returns calculated. Shape: (1258, 98)
Log returns calculated. Shape: (1257, 98)
